In [61]:
import sedona.db
import os

bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

In [3]:
sd.sql("SELECT ST_Point(0, 1) as geom").show()

┌────────────┐
│    geom    │
│  geometry  │
╞════════════╡
│ POINT(0 1) │
└────────────┘


In [59]:
buildings = sd.read_parquet(
    f"s3://{bucket_name}/source_data/sedonadb/buildings/buildings.parquet"
)

In [60]:
places = sd.read_parquet(
    f"s3://{bucket_name}/source_data/sedonadb/buildings/buildings.parquet"
)

In [14]:
buildings.to_view("buildings")
places.to_view("places")

In [ ]:
# Spatial join

In [18]:
sd.sql(
"""
SELECT
    b.id, p.id
FROM buildings AS b
JOIN places AS p ON ST_Intersects(b.geometry, p.geometry)

"""
).show()

┌──────────────────────────────────────┬──────────────────────────────────────┐
│                  id                  ┆                  id                  │
│                 utf8                 ┆                 utf8                 │
╞══════════════════════════════════════╪══════════════════════════════════════╡
│ 4c400aca-8476-4553-bc9b-4ee0019380f5 ┆ 4c400aca-8476-4553-bc9b-4ee0019380f5 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ ae2eb32b-926a-44c2-aa47-048d16b471d8 ┆ ae2eb32b-926a-44c2-aa47-048d16b471d8 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ ece9fdaf-a1a2-4f84-b77c-c670e97d1fc4 ┆ ece9fdaf-a1a2-4f84-b77c-c670e97d1fc4 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ 9f2849f4-74ff-436d-a3de-fa6aa9da4faf ┆ 9f2849f4-74ff-436d-a3de-fa6aa9da4faf │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ c6d187c7-dff2-42a0-9a91-4a4b7449ae4d ┆

In [19]:
# spatial join with aggregation

In [22]:
sd.sql(
"""
WITH intersected AS (
    SELECT
        b.id,
        b.geometry,
        p.id AS p_id
    FROM buildings AS b
    LEFT JOIN places AS p ON ST_Intersects(b.geometry, p.geometry)
)
SELECT 
    id,
    FIRST_VALUE(geometry),
    count(*) AS cnt
FROM intersected
GROUP BY id
ORDER BY cnt DESC
"""
).show()

┌──────────────────────────────────────┬───────────────────────────────────────────────────┬───────┐
│                  id                  ┆         first_value(intersected.geometry)         ┆  cnt  │
│                 utf8                 ┆                       binary                      ┆ int64 │
╞══════════════════════════════════════╪═══════════════════════════════════════════════════╪═══════╡
│ 2a1940ce-62bb-4db8-ad66-2a409afe068f ┆ 00000000030000000100000061402ccde78910b55a404909… ┆    15 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ 886fa83b-ce4e-4419-8744-b1c3ebab6e5b ┆ 00000000030000000100000015402d068af4bb2860404911… ┆    15 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ 2c3cd1fc-a713-4f52-b7a7-7a7dad3e3d11 ┆ 0000000003000000020000005f402d0f39cafce8f4404904… ┆    15 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌

In [25]:
# spatial join with transformation to metric system, searching in 500 m radius

In [24]:
sd.sql(
"""
WITH intersected AS (
    SELECT
        b.id,
        b.geometry,
        p.id AS p_id
    FROM buildings AS b
    LEFT JOIN places AS p ON ST_DWithin(ST_Transform(b.geometry, 'EPSG:5514'), ST_Transform(p.geometry, 'EPSG:5514'), 500)
)
SELECT 
    id,
    FIRST_VALUE(geometry),
    count(*) AS cnt
FROM intersected
GROUP BY id
ORDER BY cnt DESC
"""
).show()


┌──────────────────────────────────────┬───────────────────────────────────────────────────┬───────┐
│                  id                  ┆         first_value(intersected.geometry)         ┆  cnt  │
│                 utf8                 ┆                       binary                      ┆ int64 │
╞══════════════════════════════════════╪═══════════════════════════════════════════════════╪═══════╡
│ bbefb3fe-5c7e-460c-837f-482d001e51fd ┆ 00000000030000000100000007402cf4ad0734a7a7404905… ┆  1840 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ d8547e3a-d687-45c8-a50a-15cd199b94da ┆ 0000000003000000010000000a402cf4b7c6ab4a9c404905… ┆  1837 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌┤
│ f2a2ffe9-3135-4900-a0a0-cb1cd48af3d9 ┆ 00000000030000000100000009402cf49c28c012aa404905… ┆  1822 │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌

In [5]:
# interoperability with geopandas

In [31]:
import geopandas as gpd
from dataclasses import dataclass
from shapely.geometry import Point

In [35]:
@dataclass
class Observation:
    idx: int
    geometry: Point


In [46]:
# from geopandas to the sedona db

gdf = gpd.GeoDataFrame(
    [
        Observation(1, Point(0, 0)),
        Observation(2, Point(1, 0)),
        Observation(3, Point(1, 2)),
        Observation(4, Point(4, 2))
    ]
)

In [41]:
df = sd.create_data_frame(gdf)

In [42]:
df.show()

┌───────┬────────────┐
│  idx  ┆  geometry  │
│ int64 ┆  geometry  │
╞═══════╪════════════╡
│     1 ┆ POINT(0 0) │
├╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌┤
│     2 ┆ POINT(1 0) │
├╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌┤
│     3 ┆ POINT(1 2) │
├╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌┤
│     4 ┆ POINT(4 2) │
└───────┴────────────┘


In [47]:
# from sedona db to geopandas

In [49]:
cities_gdf = gpd.GeoDataFrame.from_arrow(cities.to_arrow_table())

In [55]:
cities_gdf[["id", "geometry", "class"]].loc[:5, :]

,id,geometry,class
0,4c400aca-8476-4553-bc9b-4ee0019380f5,"POLYGON ((14.30065 50.01531, 14.30065 50.01534...",None
1,ae2eb32b-926a-44c2-aa47-048d16b471d8,"POLYGON ((14.30063 50.01543, 14.30052 50.01543...",house
2,ece9fdaf-a1a2-4f84-b77c-c670e97d1fc4,"POLYGON ((14.3005 50.01562, 14.30051 50.01554,...",house
3,9f2849f4-74ff-436d-a3de-fa6aa9da4faf,"POLYGON ((14.30001 50.01586, 14.30001 50.0158,...",house
4,c6d187c7-dff2-42a0-9a91-4a4b7449ae4d,"POLYGON ((14.30051 50.01582, 14.30052 50.01572...",house
5,c08c3b6e-f9aa-4b79-9923-d81c537923c2,"POLYGON ((14.30112 50.0157, 14.30112 50.01574,...",None


In [ ]:
# interoperability with Apache Sedona Spark

In [56]:
from sedona.spark import SedonaContext

In [65]:
config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())
sedona.sparkContext.setLogLevel("ERROR")

In [ ]:
sedona

In [67]:
places_sedona_spark = sedona.read.format("geoparquet").load(
    f"s3a://{bucket_name}/source_data/sedonadb/places/places.parquet"
)

In [ ]:
# sedona spark to the sedona db

In [72]:
from sedona.spark import dataframe_to_arrow

In [73]:
places_sedona_db = sd.create_data_frame(dataframe_to_arrow(places_sedona_spark))

In [75]:
places_sedona_db.show()

┌─────────────────┬────────────────┬────────────────┬───┬────────┬────────────────┬────────────────┐
│        id       ┆    geometry    ┆      bbox      ┆ … ┆  brand ┆    addresses   ┆ operating_stat │
│       utf8      ┆    geometry    ┆     struct     ┆   ┆ struct ┆      list      ┆       us…      │
╞═════════════════╪════════════════╪════════════════╪═══╪════════╪════════════════╪════════════════╡
│ 8d886f75-ba2a-… ┆ POINT(14.2106… ┆ {xmin: 14.210… ┆ … ┆        ┆ [{freeform: M… ┆ open           │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌┼╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ 5b1d55d6-3115-… ┆ POINT(14.2130… ┆ {xmin: 14.213… ┆ … ┆        ┆ [{freeform: M… ┆ open           │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌┼╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┤
│ 332a3d46-6628-… ┆ POINT(14.2333… ┆ {xmin: 14.233… ┆ … ┆        ┆ [{freeform: ,… ┆ open           │
├╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌┼╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌╌┼╌╌╌╌╌╌╌╌

In [76]:
# sedona db to the sedona spark - not supported now